# Vier LLM-APIs vergleichen

Wie beantworten verschiedene Modelle dieselbe Frage?
Wir stellen eine Frage zu einer Schweizer Volksinitiative, zuerst auf Englisch und danach auf Deutsch.

Du schreibst vier kurze API-Funktionen. Die Prompts und der Code für den Vergleich sind vorgegeben.
Später ergänzt du einen Quelltext und probierst die OpenAI Decisions API aus.

Bearbeite einen Abschnitt nach dem anderen. Vervollständige jedes `TODO`, sobald du dort ankommst.
Du kannst deine Lösung mit `llm_intro_solution.ipynb` vergleichen.

Wir verwenden die historische Initiative **«Für Ehe und Familie – gegen die Heiratsstrafe»**,
über die am **28. Februar 2016** abgestimmt wurde. Gib dieses Datum bei deinen Fragen an,
um Verwechslungen mit späteren Vorlagen zur Ehepaarbesteuerung zu vermeiden.

### API-Keys eintragen und das Notebook öffnen

Kopiere `.env.example` in eine neue Datei namens `.env`.
Trage die vier API-Keys ein. Halte diese Datei geheim.
Die Modellnamen sind bereits vorgegeben.

Starte JupyterLab:

```sh
python -m jupyterlab
```

Öffne `llm_intro.ipynb` und wähle den Python-Kernel von `.venv` aus.
Führe die Zellen der Reihe nach aus.

### Libraries und Einstellungen laden

Führe die nächsten beiden Zellen aus. Sie laden die API-Keys und Modellnamen aus `.env`.
Du musst diesen Code nicht ändern.

Prüfe, ob bei jedem Provider `key configured: True` ausgegeben wird, bevor du fortfährst.

In [10]:
import os

import requests
import anthropic
from dotenv import load_dotenv
from google import genai
from openai import OpenAI

In [11]:
load_dotenv(".env")

google_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
deepinfra_api_key = os.getenv("DEEPINFRA_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

google_model = os.getenv("GOOGLE_MODEL", "gemini-3.8-flash")
openai_model = os.getenv("OPENAI_MODEL", "gpt-6-luna")
deepinfra_model = os.getenv("DEEPINFRA_MODEL", "deepseek-ai/DeepSeek-V4.1-Flash")
anthropic_model = os.getenv("ANTHROPIC_MODEL", "claude-haiku-5-5")

# Show whether keys are present, without printing the keys themselves.
for provider, key in [("Google", google_api_key), ("OpenAI", openai_api_key),
                      ("DeepInfra", deepinfra_api_key), ("Anthropic", anthropic_api_key)]:
    print(provider, "key configured:", bool(key))

Google key configured: True
OpenAI key configured: True
DeepInfra key configured: True
Anthropic key configured: True


## Unsere Frage

Lies die beiden Prompts unten. Sie stellen dieselbe Frage auf Englisch und Deutsch.
Führe die Zelle aus, um sie als `prompt_en` und `prompt_de` zu speichern.

Wir verwenden diese Variablen erneut, damit jeder Provider innerhalb einer Sprache
genau denselben Text erhält.

In [12]:
prompt_en = """Explain the Swiss popular initiative "Für Ehe und Familie – gegen die
Heiratsstrafe", voted on 28 February 2016, to someone unfamiliar with it.
What did it propose about taxes, social insurance and the definition of marriage?
Answer neutrally in English, using at most 120 words. Say if you do not know a detail."""

prompt_de = """Erkläre die Schweizer Volksinitiative «Für Ehe und Familie – gegen die
Heiratsstrafe», über die am 28. Februar 2016 abgestimmt wurde, für jemanden ohne Vorwissen.
Was schlug sie zu Steuern, Sozialversicherungen und zur Definition der Ehe vor?
Antworte neutral auf Deutsch mit höchstens 120 Wörtern. Sage, wenn du ein Detail nicht weisst."""

## Die vier Provider anbinden

Jede Funktion folgt demselben Muster:

1. Erstelle einen Client mit dem API-Key.
2. Sende den Prompt an das ausgewählte Modell.
3. Gib die Antwort als String zurück.

Mit `return` können wir die Antwort speichern und später vergleichen. Der Vergleichscode gibt sie aus.

### 1. Google Gemini

Übergib den Prompt als `contents`. Die Antwort steht in `response.text`.

[API-Beispiele](https://ai.google.dev/api/generate-content)

Vervollständige unten `ask_google(prompt)`.

1. Erstelle den Client mit `genai.Client(api_key=google_api_key)`.
2. Rufe `client.models.generate_content(...)` mit `model=google_model`
   und `contents=prompt` auf. Speichere das Ergebnis als `response`.
3. Gib `response.text` zurück.

In [13]:
def ask_google(prompt):
    # TODO: send prompt to this provider and return the answer as a string.
    client = genai.Client(api_key=google_api_key)
    response = client.models.generate_content(model=google_model, contents=prompt)
    return response.text

### 2. OpenAI

Hier verwenden wir die Responses API. Übergib den Prompt als `input`.
Die Antwort steht in `response.output_text`.

[API-Beispiele](https://developers.openai.com/api/docs/guides/text)

Vervollständige unten `ask_openai(prompt)`.

1. Erstelle den Client mit `OpenAI(api_key=openai_api_key)`.
2. Rufe `client.responses.create(...)` mit `model=openai_model`
   und `input=prompt` auf. Speichere das Ergebnis als `response`.
3. Gib `response.output_text` zurück.

In [31]:
def ask_openai(prompt):
    """Send prompt to this provider and return the answer as a string."""
    client = OpenAI(api_key=openai_api_key)
    response = client.responses.create(model=openai_model, input=prompt)
    return response.output_text

### 3. DeepInfra

DeepInfra funktioniert mit dem OpenAI-Python-Package, verwendet aber einen eigenen API-Key und eine eigene API-Adresse.
Dieser Aufruf sendet den Prompt als User Message.

[API-Beispiele](https://docs.deepinfra.com/chat/overview)

Vervollständige unten `ask_deepinfra(prompt)`.

1. Erstelle einen `OpenAI`-Client mit `api_key=deepinfra_api_key` und
   `base_url="https://api.deepinfra.com/v1/openai"`.
2. Rufe `client.chat.completions.create(...)` mit `model=deepinfra_model`
   und der unten stehenden User Message auf. Speichere das Ergebnis als `response`.

```python
messages=[{"role": "user", "content": prompt}]
```

3. Gib `response.choices[0].message.content` zurück. Das ist der Text der ersten Antwort.

In [15]:
def ask_deepinfra(prompt):
    """Send prompt to this provider and return the answer as a string."""
    client = OpenAI(api_key=deepinfra_api_key, base_url="https://api.deepinfra.com/v1/openai")
    response = client.chat.completions.create(model=deepinfra_model, messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

### 4. Anthropic

Anthropic erhält ebenfalls eine User Message. Die Antwort wird in mehreren Blöcken zurückgegeben.
Behalte die Blöcke vom Typ `"text"` und füge ihre Texte zu einem String zusammen.

[API-Beispiele](https://platform.claude.com/docs/en/api/messages/create)

Vervollständige unten `ask_anthropic(prompt)`.

1. Erstelle den Client mit `anthropic.Anthropic(api_key=anthropic_api_key)`.
2. Rufe `client.messages.create(...)` mit `model=anthropic_model`,
   `max_tokens=4096` und der unten stehenden User Message auf. Speichere das Ergebnis als `response`.

```python
messages=[{"role": "user", "content": prompt}]
```

3. Sammle `block.text` aus den Blöcken in `response.content`,
   für die `block.type == "text"` gilt.
4. Verbinde diese Strings mit `"\n"` und gib das Ergebnis zurück.

`max_tokens=4096` legt das Output-Budget fest. Es lässt Platz für das
interne Reasoning des Modells und für seine kurze Antwort.

In [16]:
def ask_anthropic(prompt):
    """Send prompt to this provider and return the answer as a string."""
    client = anthropic.Anthropic(api_key=anthropic_api_key)
    response = client.messages.create(model=anthropic_model, max_tokens=4096, messages=[{"role": "user", "content": prompt}])
    text = "".join(
        block.text
        for block in response.content
        if block.type == "text"
    )  
    return text

## Die Antworten vergleichen

Die folgende Funktion ist vorgegeben. Sie sendet `prompt` an jede der vier API-Funktionen
und gibt deren Antworten aus.

Sie benötigt nur ein Argument: den Prompt-Text. Jede API-Funktion wählt ihr Modell bereits selbst aus.
Zum Beispiel verwendet `compare_models(prompt_en)` den englischen Prompt.

Führe diese Zelle aus, um die Funktion zu definieren. Requests werden erst gesendet, wenn wir sie weiter unten aufrufen.

In [17]:
def compare_models(prompt):
    answers = {}

    print("\nGoogle Gemini")
    answers["Google Gemini"] = ask_google(prompt)
    print(answers["Google Gemini"])

    print("\nOpenAI")
    answers["OpenAI"] = ask_openai(prompt)
    print(answers["OpenAI"])

    print("\nDeepInfra")
    answers["DeepInfra"] = ask_deepinfra(prompt)
    print(answers["DeepInfra"])

    print("\nAnthropic")
    answers["Anthropic"] = ask_anthropic(prompt)
    print(answers["Anthropic"])

    return answers

### A. Ohne Quelltext fragen

Beginne mit Englisch. Führe die nächste Zelle aus und lies die vier Antworten, bevor du fortfährst.

Jeder Vergleich sendet vier API-Requests. Wenn du eine Zelle erneut ausführst, werden diese nochmals gesendet
und können Kosten verursachen. Das gesamte Notebook sendet 26 Requests.

In [32]:
answers_en = compare_models(prompt_en)


Google Gemini
According to official Swiss government voting materials, the popular initiative "For marriage and family – against the marriage penalty," launched by the Christian Democratic People's Party (CVP), sought to amend the Swiss Federal Constitution regarding marriage.

Regarding taxes and social insurance, the initiative aimed to eliminate financial disadvantages for married couples. Under the existing system, spouses' incomes were combined, sometimes pushing them into higher tax brackets and capping combined pension benefits (AHV) compared to unmarried cohabiting couples. The proposal required that married couples not be economically disadvantaged compared to other lifestyles. 

Regarding the definition of marriage, the initiative proposed explicitly defining it in the constitution as the legally regulated union between a man and a woman, which critics argued would prevent the legalization of same-sex marriage. Voters narrowly rejected the initiative in 2016.

OpenAI
The ini

Führe nun die deutsche Version aus. Beschreiben die Modelle in beiden Sprachen dieselben Vorschläge?

In [33]:
answers_de = compare_models(prompt_de)


Google Gemini
Gemäss den amtlichen Abstimmungserläuterungen der Schweizer Bundeskanzlei und der Initiantin, der damaligen CVP, zielte die Initiative darauf ab, finanzielle Nachteile für Verheiratete zu beseitigen und die Ehe verfassungsrechtlich festzuschreiben:

* **Steuern:** Verheiratete Paare sollten bei der direkten Bundessteuer gegenüber unverheirateten Paaren in gleicher wirtschaftlicher Lage nicht mehr benachteiligt werden (Abschaffung der sogenannten «Heiratsstrafe»).
* **Sozialversicherungen:** Auch bei den Sozialversicherungen (etwa der AHV, bei der die Renten von Ehepaaren gedeckelt sind) durfte die Ehe zu keinen finanziellen Schlechterstellungen führen.
* **Definition der Ehe:** Die Vorlage sah vor, die Ehe in der Bundesverfassung ausdrücklich als die auf Dauer angelegte Lebensgemeinschaft von Mann und Frau zu definieren.

Befürworter argumentierten, die Initiative schaffe endlich steuerliche Gerechtigkeit für Familien. Gegner kritisierten vor allem die vorgeschlagene Ehe

### B. Den Modellen einen Quelltext geben

Lies die beiden Snippets unten. Sie fassen die Vorlage zusammen, über die 2016 abgestimmt wurde.
Sie beschreiben weder die aktuelle Rechtslage noch nennen sie Abstimmungsergebnisse.

Diese Zusammenfassungen für die Übung basieren auf der
[Beschreibung des Bundesrats](https://www.admin.ch/de/fur-ehe-und-familie).

In [20]:
source_text_en = """The 2016 initiative sought to prevent disadvantages for married
couples in taxation and social insurance compared with other living arrangements.
It proposed treating spouses as one economic unit for tax purposes and defining
marriage as a lasting union between a man and a woman. The Federal Council and
Parliament recommended rejecting the initiative."""

source_text_de = """Die Initiative von 2016 wollte Benachteiligungen von Ehepaaren
bei Steuern und Sozialversicherungen gegenüber anderen Lebensformen verhindern.
Sie wollte Ehepaare steuerlich als eine Wirtschaftsgemeinschaft behandeln und die
Ehe als dauerhafte Verbindung von Mann und Frau definieren. Bundesrat und Parlament
empfahlen, die Initiative abzulehnen."""

Wir behalten die ursprüngliche Frage bei und ergänzen das passende Snippet.
Ausserdem weisen wir die Modelle an, nur den bereitgestellten Text zu verwenden.

Führe die nächste Zelle aus und lies den vollständigen englischen Prompt, der ausgegeben wird.

In [21]:
prompt_with_text_en = (
    f"{prompt_en}\n\n"
    "Use only the source text below. If a detail is missing, say it is not in the text.\n\n"
    f"Source text:\n{source_text_en}"
)

prompt_with_text_de = (
    f"{prompt_de}\n\n"
    "Verwende nur den folgenden Quelltext. Wenn ein Detail fehlt, sage, dass es nicht im Text steht.\n\n"
    f"Quelltext:\n{source_text_de}"
)

print(prompt_with_text_en)

Explain the Swiss popular initiative "Für Ehe und Familie – gegen die
Heiratsstrafe", voted on 28 February 2016, to someone unfamiliar with it.
What did it propose about taxes, social insurance and the definition of marriage?
Answer neutrally in English, using at most 120 words. Say if you do not know a detail.

Use only the source text below. If a detail is missing, say it is not in the text.

Source text:
The 2016 initiative sought to prevent disadvantages for married
couples in taxation and social insurance compared with other living arrangements.
It proposed treating spouses as one economic unit for tax purposes and defining
marriage as a lasting union between a man and a woman. The Federal Council and
Parliament recommended rejecting the initiative.


Führe den englischen Vergleich aus. Welche Details aus den Antworten findest du im Snippet wieder?

In [34]:
answers_with_text_en = compare_models(prompt_with_text_en)


Google Gemini
The 2016 Swiss popular initiative aimed to prevent disadvantages for married couples compared with other living arrangements. 

Based on the provided text, it proposed the following:

* **Taxes:** It sought to prevent tax disadvantages for married couples and proposed treating spouses as one economic unit for tax purposes.
* **Social insurance:** It proposed preventing disadvantages for married couples compared to other living arrangements; further specific details regarding social insurance are not mentioned in the text.
* **Definition of marriage:** It proposed defining marriage as a lasting union between a man and a woman. 

The text also notes that the Federal Council and Parliament recommended rejecting the initiative.

OpenAI
The initiative aimed to prevent married couples from being disadvantaged in taxation and social insurance compared with people in other living arrangements. It proposed treating spouses as a single economic unit for tax purposes and defining m

Wiederhole den Vergleich mit dem deutschen Prompt und Snippet.

In [35]:
answers_with_text_de = compare_models(prompt_with_text_de)


Google Gemini
Die Volksinitiative von 2016 schlug Folgendes vor:

* **Steuern:** Sie wollte Benachteiligungen von Ehepaaren gegenüber anderen Lebensformen verhindern. Ehepaare sollten steuerlich als eine gemeinsame Wirtschaftsgemeinschaft behandelt werden.
* **Sozialversicherungen:** Auch hier sollten Benachteiligungen von Ehepaaren im Vergleich zu anderen Lebensformen beseitigt werden. Konkretere Details dazu werden im Text nicht genannt.
* **Definition der Ehe:** Die Ehe sollte als eine dauerhafte Verbindung zwischen Mann und Frau definiert werden.

Bundesrat und Parlament empfahlen die Ablehnung der Initiative. Weitere Details gehen aus dem Text nicht hervor.

OpenAI
Die Initiative wollte Benachteiligungen von Ehepaaren gegenüber anderen Lebensformen bei Steuern und Sozialversicherungen verhindern. Bei den Steuern sollten Ehepaare als eine Wirtschaftsgemeinschaft behandelt werden. Zudem sollte die Ehe als dauerhafte Verbindung zwischen Mann und Frau definiert werden. Bundesrat und 

### Wie uns das auf RAG vorbereitet

Hier haben wir selbst eine Textpassage ausgewählt und in den Prompt eingefügt.

Später wird **retrieval-augmented generation (RAG)** relevante Textpassagen in Dokumenten
finden und dem Prompt hinzufügen. Das Modell beantwortet die Frage dann anhand dieser Informationen.
Diesen letzten Schritt haben wir gerade geübt.

### C. Nach fehlenden Informationen fragen

Die Snippets nennen den Anteil der Ja-Stimmen nicht.
Was sollte ein Modell antworten, wenn es ausschliesslich diesen Text verwenden darf?

Lies die beiden Prompts unten und führe dann die Zelle aus, um sie zu speichern.

In [24]:
missing_prompt_en = (
    "What percentage voted yes on the initiative on 28 February 2016? "
    "Answer in English using only the source text. "
    "If the answer is missing, say it is not in the text.\n\n"
    f"Source text:\n{source_text_en}"
)

missing_prompt_de = (
    "Wie hoch war der Ja-Stimmenanteil zur Initiative am 28. Februar 2016? "
    "Antworte auf Deutsch und verwende nur den Quelltext. "
    "Wenn die Antwort fehlt, sage, dass sie nicht im Text steht.\n\n"
    f"Quelltext:\n{source_text_de}"
)

Probiere die englische Frage aus. Prüfe, ob jede Antwort darauf hinweist, dass die Prozentangabe fehlt.

In [37]:
missing_answers_en = compare_models(missing_prompt_en)


Google Gemini
This information is not in the text.

OpenAI
The percentage who voted yes is not in the text.

DeepInfra
It is not in the text.

Anthropic
The source text does not say what percentage voted yes on the initiative on 28 February 2016. It only describes the initiative's aims (preventing disadvantages for married couples in taxation and social insurance, treating spouses as one economic unit for tax purposes, and defining marriage as a lasting union between a man and a woman) and notes that the Federal Council and Parliament recommended rejecting it. The voting result is not in the text.


Probiere es nun auf Deutsch. Nennt ein Modell eine Zahl, obwohl es angewiesen wurde, nur den Text zu verwenden?

In [36]:
missing_answers_de = compare_models(missing_prompt_de)


Google Gemini
Diese Information steht nicht im Text.

OpenAI
Der Ja-Stimmenanteil steht nicht im Text.

DeepInfra
Die Antwort steht nicht im Text.

Anthropic
Im Quelltext steht nicht, wie hoch der Ja-Stimmenanteil war.


## Innehalten und vergleichen

Schreibe in jede Spalte eine kurze Notiz. Erwähne Unterschiede zwischen Englisch und Deutsch,
wenn dir welche auffallen.

| Provider | Ohne Text | Mit Text | Fehlende Informationen |
| --- | --- | --- | --- |
| Google Gemini | | | |
| OpenAI | | | |
| DeepInfra | | | |
| Anthropic | | | |

Diskutiert diese Fragen:

- Was hat sich verändert, als du den Text bereitgestellt hast?
- Haben sich durch den Sprachwechsel Fakten verändert?
- Haben die Modelle zugegeben, wenn Informationen fehlten?
- Welche Antworten würdest du überprüfen, bevor du sie mit anderen teilst?

Du kannst eine gespeicherte Antwort erneut lesen, ohne einen weiteren Request zu senden:

```python
answers_with_text_en["DeepInfra"]
```

Wenn ein Aufruf fehlschlägt, prüfe den API-Key, das Guthaben und den Modellzugriff des betreffenden Providers.
Führe danach die Vergleichszelle erneut aus.

## Eine weitere API: OpenAI Decisions

Bisher haben wir Modelle gebeten, Antworten zu formulieren. Nun stellen wir zwei Ja/Nein-Fragen
zu einem Snippet:

1. Steht darin, dass Ehepaare steuerlich als eine wirtschaftliche Einheit behandelt würden?
2. Wird darin der Anteil der Ja-Stimmen genannt?

Die [Decisions API](https://developers.openai.com/api/docs/guides/decisions)
kann für jede Frage eine Wahrscheinlichkeit zurückgeben. Ein Wert nahe 1 bedeutet, dass das Modell
die Antwort für Ja hält; ein Wert nahe 0 bedeutet, dass es die Antwort für Nein hält.

Diese Art von Prüfung heisst `predicate`. Lies die beiden Fragen unten und führe die Zelle aus.
Für beide Snippets werden dieselben Fragen verwendet.

In [27]:
decision_questions = [
    {
        "type": "predicate",
        "name": "tax_unit",
        "instructions": (
            "Does the text say the initiative proposed treating spouses "
            "as one economic unit for tax purposes?"
        ),
    },
    {
        "type": "predicate",
        "name": "vote_percentage",
        "instructions": (
            "Does the text give the percentage who voted yes? "
            "Use only the supplied text."
        ),
    },
]

### Das Snippet und die Fragen senden

Mit `requests.post` senden wir einen JSON-Request direkt an die API.
Das funktioniert auch mit Versionen des OpenAI SDK, die Decisions noch nicht enthalten.
Der Request verwendet denselben OpenAI-Key wie zuvor.

Verwende für diesen Endpoint `gpt-6-luna`. Die Funktion gibt eine Liste von Antwort-Dictionaries zurück.
Ihr Ergebnis hat deshalb ein anderes Format als das der vier Funktionen zur Text Generation.

Vervollständige unten `ask_openai_decisions(source_text)`.

Rufe `requests.post(...)` mit diesen Argumenten auf:

- URL: `"https://api.openai.com/v1/decisions"`
- `headers`: `{"Authorization": f"Bearer {openai_api_key}"}`
- `timeout`: `60` Sekunden
- `json`: das unten gezeigte Dictionary

```python
{
    "model": "gpt-6-luna",
    "input": source_text,
    "questions": decision_questions,
}
```

Speichere das Ergebnis als `response`. Danach:

1. Rufe `response.raise_for_status()` auf, damit ein fehlgeschlagener Request als Fehler gemeldet wird.
2. Gib `response.json()["answers"]` zurück, um die Liste der Antworten zu erhalten.

In [28]:
def ask_openai_decisions(source_text):
    # TODO: send the snippet and both predicate questions to the Decisions API.
    # Return the list of answers from the JSON response.
    response = requests.post(url="https://api.openai.com/v1/decisions", 
                  headers={"Authorization": f"Bearer {openai_api_key}"}, 
                  timeout=60,
                  json={
                    "model": "gpt-6-luna",
                    "input": source_text,
                    "questions": decision_questions,
                })
    response.raise_for_status()
    return response.json()["answers"]

### Die Ergebnisse lesen

Überlege vor dem Ausführen des englischen Snippets, welche Frage eine hohe Wahrscheinlichkeit erhalten sollte.
Führe dann die Zelle aus. Jedes ausgegebene Dictionary entspricht einer Antwort:

- `name` identifiziert die Frage.
- `type` sollte `predicate` sein.
- `probability` ist die Schätzung des Modells.

Wenn der Typ `refusal` ist, hat die API die Prüfung abgelehnt und keine Wahrscheinlichkeit angegeben.

In [29]:
decision_answers_en = ask_openai_decisions(source_text_en)
for answer in decision_answers_en:
    print(answer)

{'type': 'predicate', 'name': 'tax_unit', 'probability': 1.0}
{'type': 'predicate', 'name': 'vote_percentage', 'probability': 1.0}


Wiederhole dies mit dem deutschen Snippet. Stimmen die Ergebnisse mit deinem Verständnis beider Texte überein?
Eine Wahrscheinlichkeit ist weiterhin eine Modellschätzung. Überprüfe sie deshalb anhand des Snippets.

In [30]:
decision_answers_de = ask_openai_decisions(source_text_de)
for answer in decision_answers_de:
    print(answer)

{'type': 'predicate', 'name': 'tax_unit', 'probability': 1.0}
{'type': 'predicate', 'name': 'vote_percentage', 'probability': 0.91}
